In [17]:
import pathlib
import ollama
from itertools import product
import re
import tiktoken
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

In [18]:
OUTPUT_MD = '../data/output.md'
PROMPTS = '../prompts/prompts.txt'
NON_PERSISTENT_PROMPTS = '../prompts/non_persistent_prompts.txt'
LLM_CONTEXT = '../prompts/llm_context.txt'

In [19]:
system_prompt = """
Role:
Experiences racing driver coach.

Task:
Transform the tabular telemetry data into clear,
readable coaching feedback describing how this track segment was driven compared to an optimally driven reference lap. Each attribute contains two values: First value, driver; second value reference.
Explain how the segment unfolded in driving terms and identify errors if present.

At the end, output a json object with the following structure, where status signals wether the message has to be read out (has only to be read out when significant improvements are necessary; status can be True or False) and the message states the improvement text:
{
    'status': ...,
    'message': ...,
}
Only output the json!

Notes:
- Consider the units, but never mention them explicitly.
- Translate telemetry only into: throttle, braking, steering, speed, track position.
- Do not mention axes, telemetry terms, or timestamps.
- Think in track sections, not time.
- Focus on cause and effect.
- Do not speculate beyond the data.

Examples:
Good throttle use, speed carried well.
Clean line, no time lost here.
Brake earlier to stabilize the car.
Turn in earlier for better exit.
Strong exit, close to optimal.
Over-slowed entry cost exit speed.

Negative examples:
- The strongly negative acceleration_x (-34 vs. reference 0) together with a high Z value…
- From about 9 s onward you reach lateral accelerations of up to +20 m/s²…
- Excessive negative acceleration_x and incorrectly dosed yaw rotation dominate…
"""

user_prompt = ""

In [20]:
# Count tokens for System prompt and User prompt
def count_tokens(prompt: str, model_name: str = "gpt-4") -> int:
    """
    Zählt die Tokens für einen Prompt für das angegebene Modell.
    """
    encoding = tiktoken.encoding_for_model(model_name)
    tokens = encoding.encode(prompt)
    return len(tokens)

text = []

with open('../data/output.md', "r", encoding="utf-8") as f:
    for line in f:
        line = line.rstrip("\n")
        if line != '\'':
            text.append(line + '\n')

sys_tokens = count_tokens(system_prompt, model_name="gpt-4")  # ersetze ggf. durch dein Modell
print(f"Token-Anzahl System: {sys_tokens}")
usr_tokens = count_tokens(user_prompt, model_name="gpt-4")  # ersetze ggf. durch dein Modell
print(f"Token-Anzahl User {usr_tokens}")

Token-Anzahl System: 316
Token-Anzahl User 0


In [21]:
# Helper functions
def log_response(timestamps, lap, segment, system_prompt, user_prompt, resp, md, segment_info):
    # -- Logging ---
    if(segment == 0):
        log_round(system_prompt, user_prompt)
    log_segment_response(timestamps, lap, segment, resp, md, segment_info)

def log_round(system_prompt, user_prompt):
    # --- Logging ---
    pathlib.Path("prompts").mkdir(parents=True, exist_ok=True)
    with open(PROMPTS, "a", encoding="utf-8") as file:
        file.writelines([
            "XXXX" * 80 + "\n",
            "System Prompt: " + system_prompt + "\n",
            "User Prompt: " + user_prompt + "\n",
        ])
    with open(NON_PERSISTENT_PROMPTS, "a", encoding="utf-8") as file:
        file.writelines([
            "XXXX" * 80 + "\n",
            "System Prompt: " + system_prompt + "\n",
            "User Prompt: " + user_prompt + "\n",
        ])

def log_segment_response(timestamps, lap, segment, resp, md, segment_info):
    min_timestamp = min(timestamps)
    max_timestamp = max(timestamps)

    pathlib.Path("prompts").mkdir(parents=True, exist_ok=True)
    with open(PROMPTS, "a", encoding="utf-8") as file:
        file.writelines([
            "__" * 80 + "\n",
            f"Lap: {lap}, Segment: {segment}, Sequence: {min_timestamp} - {max_timestamp}\n",
            f"{segment_info}\n",
            "Response: " + resp["message"]["content"] + "\n\n",
        ])
    with open(NON_PERSISTENT_PROMPTS, "a", encoding="utf-8") as file:
        file.writelines([
            "__" * 10 + "\n",
            f"Lap: {lap}, Segment: {segment}, Sequence: {min_timestamp} - {max_timestamp}\n",
            f"{md}\n",
            f"{segment_info}\n",
            "Response: " + resp["message"]["content"] + "\n\n",
        ])
    with open(LLM_CONTEXT, "a", encoding="utf-8") as file:
        file.writelines([
            f"Lap: {lap}, Segment: {segment}, Sequence: {min_timestamp} - {max_timestamp}\n",
            f"{segment_info}\n",
            "Response: " + resp["message"]["content"] + "\n\n"
        ])

def get_segment_md(md_row: str) -> int:
    return int(md_row.split("|")[-2].strip())

def get_lap_md(md_row: str) -> int:
    return int(md_row.split("|")[-3].strip())

def get_timestamp_md(md_row: str) -> float:
    ts_cell = md_row.split("|")[2].strip()
    return float(ts_cell.strip("()").split(",")[0])

# MD Table Approach

In [22]:
def get_segment_information(segment_rows)-> str:
    num = r"[-+]?(?:\d*\.\d+|\d+\.?\d*)(?:[eE][-+]?\d+)?"
    pair_re = re.compile(rf"\(\s*({num})\s*,\s*({num})\s*\)")

    # lists for user / optimal (reference)
    timestamps_user = []
    timestamps_opt = []
    speeds_user = []
    speeds_opt = []
    yaws_user = []
    yaws_opt = []

    for row in segment_rows:
        cells = [c.strip() for c in row.split("|")]

        # timestamp usually in column index 2
        if len(cells) > 2:
            m = pair_re.search(cells[2])
            if m:
                timestamps_user.append(float(m.group(1)))
                timestamps_opt.append(float(m.group(2)))

        # yaw usually in column index 6
        if len(cells) > 6:
            m = pair_re.search(cells[6])
            if m:
                yaws_user.append(float(m.group(1)))
                yaws_opt.append(float(m.group(2)))

        # speed usually in column index 10
        if len(cells) > 10:
            m = pair_re.search(cells[10])
            if m:
                speeds_user.append(float(m.group(1)))
                speeds_opt.append(float(m.group(2)))

    # --- Timestamp / duration ---
    if timestamps_user:
        start_user = min(timestamps_user)
        end_user = max(timestamps_user)
        duration_user = end_user - start_user
    else:
        start_user = end_user = duration_user = 0.0

    if timestamps_opt:
        start_opt = min(timestamps_opt)
        end_opt = max(timestamps_opt)
        duration_opt = end_opt - start_opt
    else:
        start_opt = end_opt = duration_opt = 0.0

    # --- Speed stats ---
    max_speed_user = max(speeds_user) if speeds_user else 0.0
    min_speed_user = min(speeds_user) if speeds_user else 0.0
    max_speed_opt = max(speeds_opt) if speeds_opt else 0.0
    min_speed_opt = min(speeds_opt) if speeds_opt else 0.0

    # --- Yaw stats (use absolute values for extremes) ---
    max_yaw_user = max((abs(v) for v in yaws_user), default=0.0)
    max_yaw_opt = max((abs(v) for v in yaws_opt), default=0.0)

    # --- Build summary ---
    ret = (
        f"Time — user: {duration_user:.3f}s, "
        f"ref: {duration_opt:.3f}s (Δ {duration_user - duration_opt:+.3f}s). "
    )

    if speeds_user or speeds_opt:
        ret += (
            f"Max speed — user: {max_speed_user:.1f} km/h, ref: {max_speed_opt:.1f} km/h; "
            f"Min speed — user: {min_speed_user:.1f} km/h, ref: {min_speed_opt:.1f} km/h. "
        )
    else:
        ret += "No speed data. "

    if yaws_user or yaws_opt:
        ret += f"Max yaw (abs) — user: {max_yaw_user:.3f}°, ref: {max_yaw_opt:.3f}°."
    else:
        ret += "No yaw data."

    print(ret)
    return ret

In [23]:
text = []

# --- Read markdown file ---
with open(OUTPUT_MD, "r", encoding="utf-8") as f:
    lines = [line.rstrip("\n") for line in f if line.strip()]

# --- Split header and rows ---
header = lines[0]
separator = lines[1]
rows = lines[2:]

# Delete existing non-persistent prompts log and create new empty file
pathlib.Path(NON_PERSISTENT_PROMPTS).unlink(missing_ok=True)
pathlib.Path(NON_PERSISTENT_PROMPTS).parent.mkdir(parents=True, exist_ok=True)  # sicherstellen, dass Ordner existiert
pathlib.Path(NON_PERSISTENT_PROMPTS).touch()  # erstellt leere Datei

# Delete existing LLM context log and create new empty file
pathlib.Path(LLM_CONTEXT).unlink(missing_ok=True)
pathlib.Path(LLM_CONTEXT).parent.mkdir(parents=True, exist_ok=True)  # sicherstellen, dass Ordner existiert
pathlib.Path(LLM_CONTEXT).touch()  # erstellt leere Datei

# --- Process per lap / segment ---
for lap, segment in product([0], [0, 1, 2, 3, 4]):
    print("Lap:" + str(lap) + ", Segment:" + str(segment))
    segment_rows = [r for r in rows if get_segment_md(r) == segment and get_lap_md(r) == lap]

    if not segment_rows:
        continue
    
    md_block = "\n".join([header, separator, *segment_rows])
    #print(md_block)

    segment_info = get_segment_information(segment_rows)

    messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt + f"\n```markdown\n{md_block}\n```" + f"\n\nSegment Summary:\n{segment_info}"}
        ]
    # --- LLM call ---
    final_response = ollama.chat(
        model="llama3.1",
        messages=messages,
        # think=True
    )
    
    # Logging
    data_tokens = count_tokens(md_block, model_name="gpt-4")
    print(f"Token-Anzahl User {data_tokens}")
    timestamps = [get_timestamp_md(r) for r in segment_rows]
    log_response(timestamps, lap, segment, system_prompt, user_prompt, final_response, md_block, segment_info)

Lap:0, Segment:0
Time — user: 0.004s, ref: 0.002s (Δ +0.002s). Max speed — user: 187.0 km/h, ref: 221.0 km/h; Min speed — user: 0.0 km/h, ref: 181.0 km/h. Max yaw (abs) — user: 2.560°, ref: 2.540°.
Token-Anzahl User 804
Lap:0, Segment:1
Time — user: 0.000s, ref: 0.000s (Δ +0.000s). No speed data. No yaw data.
Token-Anzahl User 761
Lap:0, Segment:2
Time — user: 0.000s, ref: 0.000s (Δ +0.000s). No speed data. No yaw data.
Token-Anzahl User 869
Lap:0, Segment:3
Time — user: 0.000s, ref: 0.000s (Δ +0.000s). No speed data. No yaw data.
Token-Anzahl User 599
Lap:0, Segment:4
Time — user: 0.000s, ref: 0.000s (Δ +0.000s). No speed data. No yaw data.
Token-Anzahl User 491
